# Feature Engineering — Credit Risk Analytics

**Date:** July 2026  
**Dataset:** Home Credit Default Risk (Kaggle)  
**Target Variable:** `TARGET` (1 = default, 0 = repaid)  

---

## Objective

Design and document every feature to be engineered for the credit risk model. Features are drawn from all 8 Home Credit tables and organized into logical groups. Each feature is documented with:

- **Business intuition** — why this feature should predict default
- **Mathematical definition** — precise formula
- **Implementation method** — table source, aggregation strategy, edge cases

No Python code is written in this design document. The specifications below feed directly into `src/preprocess.py`.

---

## Feature Inventory Overview

| Group | Source Table | Count (est.) | Theme |
|---|---|---|---|
| A — Domain Ratios | `application_train` | ~8 | Affordability and capacity signals |
| B — Artifact & Outlier Rescues | `application_train` | ~5 | Fix known data quality issues |
| C — Bureau Aggregations | `bureau` + `bureau_balance` | ~15 | Credit-bureau debt profile |
| D — Previous Application Aggregations | `previous_application` | ~12 | Past lending relationship |
| E — Installment Payment Profile | `installments_payments` | ~10 | Granular payment discipline |
| F — POS / Cash Balance Profile | `POS_CASH_balance` | ~8 | Active installment loan behavior |
| G — Credit Card Balance Profile | `credit_card_balance` | ~10 | Revolving credit behavior |
| H — Categorical Encoding | All tables | ~20+ | Convert categories to numerics |
| I — Missing-Value Indicators | All tables | ~15 | Capture missingness as signal |
| J — Interaction Features | `application_train` | ~6 | Non-linear risk boost |
| K — Feature Selection | (post-aggregation) | ~100→final | Reduce noise, improve generalisation |

**Target total after selection:** 150–250 features.

---
## Group A — Domain Ratio Features

Derived from the main `application_train` table. These capture repayment capacity and financial stress.

---

### Feature A1: Debt-to-Income Ratio (DTI)

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower who commits a large fraction of monthly income to loan payments has less buffer for unexpected expenses (job loss, medical bills). High DTI is the single most common reason for mortgage/loan denial in consumer lending. |
| **Mathematical definition** | `DTI = AMT_ANNUITY / AMT_INCOME_TOTAL` |
| **Implementation method** | Divide `AMT_ANNUITY` (monthly installment) by `AMT_INCOME_TOTAL` (annual income / 12? or raw ratio?). Home Credit stores `AMT_ANNUITY` as monthly payment and `AMT_INCOME_TOTAL` as annual income. Use: `DTI = (AMT_ANNUITY × 12) / AMT_INCOME_TOTAL` to get a true annual burden ratio. Cap at [0, 1] range. If `AMT_INCOME_TOTAL` is 0 or missing, set DTI = NaN (to be imputed in Group I). |

---

### Feature A2: Loan-to-Income Ratio (LTI)

| Aspect | Detail |
|---|---|
| **Business intuition** | The total loan amount relative to income captures the borrower's total indebtedness. A loan that is 5× annual income is riskier than one that is 1×, even if the monthly payment is manageable (due to long tenor). |
| **Mathematical definition** | `LTI = AMT_CREDIT / AMT_INCOME_TOTAL` |
| **Implementation method** | Simple division. Cap at [0, 20] (LTI > 20 suggests data error or extremely unusual case). Apply log1p transform to reduce skew. |

---

### Feature A3: Income per Household Member

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower supporting 5 dependents on a $40K salary is under more financial pressure than a single person earning the same. Household size dilutes disposable income. |
| **Mathematical definition** | `Income_per_capita = AMT_INCOME_TOTAL / (1 + CNT_FAM_MEMBERS)` |
| **Implementation method** | Use `CNT_FAM_MEMBERS` (count of family members including the applicant). The `+1` is already included in the dataset's count. If missing, treat as 1 (applicant only). |

---

### Feature A4: Annuity-to-Credit Ratio (Implied Interest Rate Proxy)

| Aspect | Detail |
|---|---|
| **Business intuition** | The ratio of the monthly payment to the total loan amount is a rough proxy for the interest rate. Higher ratios may indicate higher-risk borrowers who were charged a premium, or shorter loan terms. Both imply higher monthly burden. |
| **Mathematical definition** | `Annuity_rate = AMT_ANNUITY / AMT_CREDIT` |
| **Implementation method** | Direct division. Multiply by 100 to express as percentage of loan paid per month. This is a rough proxy; actual APR is not in the dataset. |

---

### Feature A5: External Score Composite (Mean & Min)

| Aspect | Detail |
|---|---|
| **Business intuition** | The three external credit scores (`EXT_SOURCE_1`, `2`, `3`) are individually the strongest predictors. Their average and minimum capture overall creditworthiness. The minimum is especially informative — a single bad score from one bureau drags the applicant's risk profile down. |
| **Mathematical definition** | `EXT_MEAN = mean(EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3)` and `EXT_MIN = min(EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3)` |
| **Implementation method** | Compute row-wise mean and min. If all three are missing for a row, both composites are also missing (to be flagged in Group I). If at least one is present, use available values only for the mean. |

---

### Feature A6: Employment Length Ratio

| Aspect | Detail |
|---|---|
| **Business intuition** | How long a borrower has been employed relative to their adult life captures career stability. A 22-year-old with 2 years of work is stable; a 45-year-old with 2 years of work may indicate a recent job loss or career disruption. |
| **Mathematical definition** | `Emp_len_ratio = DAYS_EMPLOYED / DAYS_BIRTH` (both negative, so ratio > 0). Multiply by -1 for clarity. |
| **Implementation method** | Convert both to absolute values first. `ratio = abs(DAYS_EMPLOYED) / abs(DAYS_BIRTH)`. Cap at [0, 1]. Values near 0 mean very recent employment (high risk). Values near 1 mean employed most of adult life (low risk). DAYS_EMPLOYED artifact (positive values) must be handled first (see Group B). |

---

### Feature A7: Age Group Binning

| Aspect | Detail |
|---|---|
| **Business intuition** | The EDA showed a non-linear relationship: young borrowers (<30) default at ~2× the rate of older borrowers (>50). A continuous age feature may not capture this plateau effect well. Binning allows the model to learn different slopes per age band. |
| **Mathematical definition** | Convert `DAYS_BIRTH` to `AGE_YEARS = floor(abs(DAYS_BIRTH) / 365)`. Then bin: [18–25], [25–30], [30–40], [40–50], [50–60], [60–70], [70+]. Missing → separate bin. |
| **Implementation method** | Use `pd.cut()` with 7 bins. Keep both the continuous age feature (for monotonic trends) and the binned version (for non-linear effects). Ordinal-encode the bins (1–7). |

---

### Feature A8: Car Ownership Age

| Aspect | Detail |
|---|---|
| **Business intuition** | The age of a borrower's car (if owned) can proxy wealth and lifestyle stability. A 15-year-old car may indicate financial strain; a 3-year-old car suggests recent financial health. Car ownership itself (binary) is also a credit-quality signal. |
| **Mathematical definition** | `CAR_AGE = OWN_CAR_AGE`. Create `HAS_CAR = 1 if OWN_CAR_AGE not NaN else 0`. |
| **Implementation method** | Missing `OWN_CAR_AGE` likely means no car (or no data). Fill missing with 0. Create a separate binary flag `HAS_CAR`. Cap `OWN_CAR_AGE` at 30 years (unreasonable beyond that). |

---
## Group B — Artifact & Outlier Rescues

Fix known data quality issues before feature engineering. These corrections prevent model corruption.

---

### Feature B1: DAYS_EMPLOYED Anomaly Flag

| Aspect | Detail |
|---|---|
| **Business intuition** | Home Credit encodes unemployed applicants as ~365,243 days (~1000 years). This is a placeholder, not a real value. A binary flag captures "unemployed" directly — a high-risk signal. |
| **Mathematical definition** | `IS_UNEMPLOYED = 1 if DAYS_EMPLOYED == 365243 else 0`. After flagging, set the anomalous values to NaN for imputation. |
| **Implementation method** | Check `DAYS_EMPLOYED == 365243` (or > 300,000). Create binary flag. Then replace those values with NaN so they are treated as missing. |

---

### Feature B2: DAYS_EMPLOYED Clean (Post-Artifact)

| Aspect | Detail |
|---|---|
| **Business intuition** | After removing the 1000-year artifact, remaining DAYS_EMPLOYED values represent genuine employment length. This is a continuous measure of job stability. |
| **Mathematical definition** | `EMP_LENGTH_YEARS = abs(DAYS_EMPLOYED) / 365`, after replacing 365243 values with NaN. Cap at 50 years. |
| **Implementation method** | Apply the anomaly flag (B1) first, then convert to positive years. Values above 50 years are capped. |

---

### Feature B3: Income Capping (Outlier Treatment)

| Aspect | Detail |
|---|---|
| **Business intuition** | Extreme incomes (e.g., $10M/year) are likely data errors or one-in-a-million cases. Tree-based models handle outliers reasonably well, but extremes can distort scaling for logistic regression and create spurious splits. |
| **Mathematical definition** | Cap at 99.9th percentile: `AMT_INCOME_TOTAL = min(AMT_INCOME_TOTAL, P99.9)`. Optionally also cap at 0.1th percentile for the lower tail. |
| **Implementation method** | Compute P99.9 and P0.1 from training data only (to avoid leakage). Apply same thresholds to test data. Document the threshold values. |

---

### Feature B4: Flagged Extremely High Loan Amounts

| Aspect | Detail |
|---|---|
| **Business intuition** | A loan amount far above the typical range for this lender's product suite may be an edge case requiring special underwriting. Flagging it separately lets the model learn if these cases are systematically riskier. |
| **Mathematical definition** | `IS_HIGH_LOAN = 1 if AMT_CREDIT > P99(AMT_CREDIT) else 0` |
| **Implementation method** | Use training-set 99th percentile as threshold. Apply to test set. |

---
## Group C — Bureau Aggregation Features

The `bureau` table contains 1+ rows per applicant — each row is a credit reported to the bureau (past loans, credit cards, mortgages from other lenders). The `bureau_balance` table contains monthly snapshots of each bureau credit.

These features capture the borrower's **total indebtedness outside of Horizon**.

---

### Feature C1: Total Number of Bureau Credits

| Aspect | Detail |
|---|---|
| **Business intuition** | A large number of past/current credits suggests a seasoned borrower with an established credit history. A very small number suggests a thin file. Both extremes carry risk. |
| **Mathematical definition** | `BUREAU_COUNT = count(SK_ID_BUREAU)` grouped by `SK_ID_CURR` |
| **Implementation method** | Group by applicant ID, count rows. No edge cases beyond applicants with zero bureau records (should be very rare). |

---

### Feature C2: Total Outstanding Debt

| Aspect | Detail |
|---|---|
| **Business intuition** | The total debt a borrower owes across all financial institutions. High total debt relative to income is a strong stress signal. |
| **Mathematical definition** | `TOTAL_DEBT = sum(AMT_CREDIT_SUM_DEBT)` grouped by `SK_ID_CURR` |
| **Implementation method** | Sum `AMT_CREDIT_SUM_DEBT` per applicant. If the column has many missing values, check for a similar column (`AMT_CREDIT_SUM_OVERDUE` may be an alternative). Consider log1p transform. |

---

### Feature C3: Total Credit Limit Across All Bureau Accounts

| Aspect | Detail |
|---|---|
| **Business intuition** | Total available credit across cards, lines of credit, and overdrafts. Combined with total debt, this allows calculation of overall utilization. |
| **Mathematical definition** | `TOTAL_CREDIT_LIMIT = sum(AMT_CREDIT_SUM_LIMIT)` grouped by `SK_ID_CURR` |
| **Implementation method** | Sum `AMT_CREDIT_SUM_LIMIT`. Only available for credit-type products (cards, lines). Missing values likely mean the credit type has no limit (e.g., installment loan). Fill missing with 0 or treat as separate category. |

---

### Feature C4: Bureau-Level Credit Utilization

| Aspect | Detail |
|---|---|
| **Business intuition** | Utilization is the ratio of debt to available limit. High utilization (>80%) is a classic predictor of default: the borrower has maxed out their available credit and likely cannot borrow more, signaling financial distress. |
| **Mathematical definition** | `BUREAU_UTILIZATION = TOTAL_DEBT / TOTAL_CREDIT_LIMIT` |
| **Implementation method** | Compute only for applicants with `TOTAL_CREDIT_LIMIT > 0`. If limit is 0 or missing, utilization is undefined (set to NaN). Cap at [0, 2] — ratios > 2 may indicate data issues or severe over-limit situations. |

---

### Feature C5: Number of Active / Closed Credits

| Aspect | Detail |
|---|---|
| **Business intuition** | Active credits represent ongoing payment obligations; closed credits represent successfully completed ones. A high number of active credits suggests the borrower is carrying significant concurrent debt. |
| **Mathematical definition** | `ACTIVE_COUNT = sum(CREDIT_ACTIVE == 'Active')`, `CLOSED_COUNT = sum(CREDIT_ACTIVE == 'Closed')` per applicant |
| **Implementation method** | Group by applicant, sum per status. Also compute `ACTIVE_RATIO = ACTIVE_COUNT / (ACTIVE_COUNT + CLOSED_COUNT)` to capture the proportion of active to total. |

---

### Feature C6: Maximum and Average Days Past Due (DPD) on Bureau Credits

| Aspect | Detail |
|---|---|
| **Business intuition** | The worst delinquency on any credit product is a powerful signal. A borrower who has been 90+ days late on *any* account is much more likely to default than one who has always paid on time. |
| **Mathematical definition** | `MAX_DPD = max(AMT_CREDIT_MAX_OVERDUE)` and `AVG_DPD = mean(AMT_CREDIT_MAX_OVERDUE)` per applicant |
| **Implementation method** | Use `AMT_CREDIT_MAX_OVERDUE` column. Group by applicant, compute max and mean. Missing values → assume 0 (no overdue recorded). Create a binary flag `HAS_OVERDUE = 1 if MAX_DPD > 0 else 0`. |

---

### Feature C7: Credit Type Diversification

| Aspect | Detail |
|---|---|
| **Business intuition** | Borrowers who use many different credit types (credit card, mortgage, consumer finance, car loan) may be more experienced with credit — or they may be over-leveraged. The count of unique credit types is a simple diversification metric. |
| **Mathematical definition** | `CREDIT_TYPE_NUNIQUE = nunique(CREDIT_TYPE)` per applicant |
| **Implementation method** | Group by applicant, count unique `CREDIT_TYPE` values. |

---
### Feature C8: Bureau Balance Aggregations (from `bureau_balance`)

| Aspect | Detail |
|---|---|
| **Business intuition** | The `bureau_balance` table has monthly statuses (0 = no DPD, 1 = DPD 1–30, 2 = DPD 31–60, etc.) for each bureau credit. The proportion of months in delinquency captures a pattern of habitual late payment. |
| **Mathematical definition** | Several aggregations: (1) `BAL_MONTHS_DELINQUENT` = count of months with STATUS in ('1','2','3','4','5'), (2) `BAL_MONTHS_TOTAL` = total months observed, (3) `BAL_DELINQUENT_RATIO = BAL_MONTHS_DELINQUENT / BAL_MONTHS_TOTAL`, (4) `BAL_WORST_STATUS = max(STATUS)` per applicant |
| **Implementation method** | Group `bureau_balance` by `SK_ID_BUREAU`, then join to bureau table, then aggregate by `SK_ID_CURR`. STATUS values: 'C' = closed (completed), 'X' = unknown, '0' = current/up-to-date, '1'–'5' = increasing delinquency. Convert STATUS to numeric (C=0, X=NaN, 0=0, 1=1, ..., 5=5). |

---

### Feature C9: Time Since Most Recent Bureau Credit

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower whose most recent credit activity was many years ago may have an outdated credit profile. Recent activity (within 1–2 years) means the bureau data reflects current financial behavior. |
| **Mathematical definition** | `MONTHS_SINCE_RECENT_BUREAU = max(DAYS_CREDIT) / 30` (most recent = closest to 0, since DAYS_CREDIT is negative). Compute as: `-max(DAYS_CREDIT) / 30`. |
| **Implementation method** | DAYS_CREDIT is negative (days before application). Take the maximum (closest to 0), negate, divide by 30 for months. |

---
## Group D — Previous Application Aggregations

The `previous_application` table contains all prior loan applications made by each applicant at Home Credit (including those that were declined, cancelled, or unpaid). This is the most powerful internal signal of borrower behavior.

---

### Feature D1: Number of Previous Applications

| Aspect | Detail |
|---|---|
| **Business intuition** | Frequent re-applications may indicate desperation (declined → reapply). A high count of previous applications (especially recently) can signal financial instability. |
| **Mathematical definition** | `PREV_APP_COUNT = count(SK_ID_PREV)` grouped by `SK_ID_CURR` |
| **Implementation method** | Straightforward row count per applicant. |

---

### Feature D2: Previous Application Approval Rate

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower whose past applications were frequently approved is likely a good risk. Frequent declines signal that prior underwriters detected problems. |
| **Mathematical definition** | `PREV_APPROVAL_RATE = count(NAME_CONTRACT_STATUS == 'Approved') / total previous applications` |
| **Implementation method** | STATUS values include 'Approved', 'Refused', 'Canceled', 'Unused offer'. 'Canceled' and 'Unused offer' may indicate the borrower was approved but chose not to proceed — ambiguous for risk. Sensitivity analysis: compute approval rate both including and excluding canceled/unused. |

---

### Feature D3: Previous Application Default Rate

| Aspect | Detail |
|---|---|
| **Business intuition** | Past default at Horizon is the single strongest predictor of future default. If a borrower has defaulted on a previous Horizon loan, they are very likely to default again. |
| **Mathematical definition** | `PREV_DEFAULT_RATE = count(NAME_CONTRACT_STATUS == 'Unpaid') / total previous applications` |
| **Implementation method** | Filter for status = 'Unpaid'. Note: this feature looks backward and may leak information if a previous application was in default *after* the current application was made. Ensure temporal consistency: only include previous applications that started before the current application. |

---

### Feature D4: Average Previous Loan Amount

| Aspect | Detail |
|---|---|
| **Business intuition** | Borrowers who have taken larger loans in the past (and repaid them) demonstrate capacity to handle higher credit. The ratio of the current loan to average previous loan captures whether the borrower is increasing or decreasing their credit exposure. |
| **Mathematical definition** | `PREV_AVG_CREDIT = mean(AMT_CREDIT)` and `PREV_MAX_CREDIT = max(AMT_CREDIT)` grouped by `SK_ID_CURR` |
| **Implementation method** | Use `AMT_CREDIT` from previous_application table. Filter to approved loans only (approved loans are the ones that had real repayment behavior). Also compute `CREDIT_GROWTH = current_credit / PREV_AVG_CREDIT`. |

---

### Feature D5: Average Days Between Application and Decision

| Aspect | Detail |
|---|---|
| **Business intuition** | A loan decision that took a long time may indicate that underwriting found issues and required additional review. Conversely, instant approvals may signal a high-quality applicant who passed automatic checks. |
| **Mathematical definition** | `PREV_AVG_DECISION_DAYS = mean(abs(DAYS_DECISION - DAYS_APPLICATION))` |
| **Implementation method** | Both `DAYS_DECISION` and `DAYS_APPLICATION` are negative offsets from the current application date. Compute `abs(DAYS_DECISION - DAYS_APPLICATION)` = days from application to decision. Group by applicant, take mean. |

---

### Feature D6: Time Since Last Previous Application

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower who last applied 3 years ago has a longer track record since their last interaction with Horizon. A borrower who applied last week may be shopping around or in urgent need of funds. |
| **Mathematical definition** | `MONTHS_SINCE_LAST_PREV = abs(max(DAYS_DECISION)) / 30` |
| **Implementation method** | Take the maximum `DAYS_DECISION` (closest to 0 = most recent), take absolute value, divide by 30. |

---

### Feature D7: Previous Application Contract Type Diversity

| Aspect | Detail |
|---|---|
| **Business intuition** | Borrowers who use a mix of cash loans and revolving (credit card) products may be more financially sophisticated. Focused borrowers (always cash loans) may be more predictable. |
| **Mathematical definition** | `PREV_CONTRACT_NUNIQUE = nunique(NAME_CONTRACT_TYPE)` per applicant |
| **Implementation method** | Group by applicant, count unique contract types. Also compute `PREV_CASH_RATIO = count(NAME_CONTRACT_TYPE == 'Cash loans') / total_prev` |

---
## Group E — Installment Payment Profile

The `installments_payments` table records every single payment made (or missed) on each previous loan. This is the most granular behavioral data available — it shows *how* the borrower repays, not just *whether* they repaid.

---

### Feature E1: Total Number of Installments (All Previous Loans)

| Aspect | Detail |
|---|---|
| **Business intuition** | The total number of installments captures the borrower's total experience with repayment. More installments = more data points on payment behavior. |
| **Mathematical definition** | `TOTAL_INSTALLMENTS = count(NUM_INSTALMENT_VERSION)` grouped by `SK_ID_CURR` |
| **Implementation method** | Join installments table to previous_application via `SK_ID_PREV`, then group by `SK_ID_CURR`. |

---

### Feature E2: Number of Late Payments

| Aspect | Detail |
|---|---|
| **Business intuition** | Each payment that was paid after the due date is a micro-signal of payment discipline. A borrower who was late on 10 out of 50 payments is clearly higher risk than one who was late on 0 out of 50. |
| **Mathematical definition** | `LATE_PAYMENTS = count(DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT)`, where both dates are relative to application date. If `DAYS_ENTRY_PAYMENT` (when the payment was actually made) > `DAYS_INSTALMENT` (scheduled date), the payment was late. |
| **Implementation method** | Compare the two date columns. Late if `DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT`. Also compute `LATE_RATIO = LATE_PAYMENTS / TOTAL_INSTALLMENTS`. |

---

### Feature E3: Average Days Late and Maximum Days Late

| Aspect | Detail |
|---|---|
| **Business intuition** | *How* late matters. A borrower who pays 1 day late every month is very different from one who pays 90 days late. The average tells us about typical behavior; the maximum reveals the worst case. |
| **Mathematical definition** | `AVG_DAYS_LATE = mean(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)` for late payments only. `MAX_DAYS_LATE = max(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)` across all payments. Only positive differences (late). For early/on-time payments, difference is <= 0. |
| **Implementation method** | Compute `PAYMENT_DIFF = DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT`. For average, filter to `PAYMENT_DIFF > 0`. For max, take max across all. Also compute `MIN_PAYMENT_DIFF` (negative = early) to capture early payers (low risk signal). |

---

### Feature E4: Number of Missed Payments (Never Paid)

| Aspect | Detail |
|---|---|
| **Business intuition** | An installment that was never paid represents a broken promise to pay. Even if the loan was eventually repaid, missed installments indicate periods of serious financial distress. |
| **Mathematical definition** | `MISSED_PAYMENTS = count(AMT_PAYMENT == 0 or AMT_PAYMENT is NaN)` where `AMT_INSTALMENT > 0` |
| **Implementation method** | Look for rows with zero or missing payment amount where payment was expected. This applies after the loan was active (not before its start date). |

---

### Feature E5: Payment Amount Consistency (Std Dev)

| Aspect | Detail |
|---|---|
| **Business intuition** | Borrowers who pay the exact amount due every month are stable. Borrowers who pay varying amounts (sometimes more, sometimes less, sometimes nothing) are less predictable. The standard deviation of payment amounts captures this volatility. |
| **Mathematical definition** | `PAYMENT_STD = std(AMT_PAYMENT)` per applicant across all installments. Normalize by average payment: `PAYMENT_CV = std / mean` (coefficient of variation). |
| **Implementation method** | Group by `SK_ID_CURR`, compute std and mean of `AMT_PAYMENT`. Use CV to make the metric scale-independent. |

---

### Feature E6: Days Past Due > 30 / 60 / 90 Flags

| Aspect | Detail |
|---|---|
| **Business intuition** | Regulatory buckets (30, 60, 90+ DPD) are standard in banking. A borrower who has ever been 90+ days late on *any* installment is a serious default risk. The count of 90+ DPD events per applicant is a hard risk signal. |
| **Mathematical definition** | Three features: `DPD30_COUNT = count(PAYMENT_DIFF > 30)`, `DPD60_COUNT = count(PAYMENT_DIFF > 60)`, `DPD90_COUNT = count(PAYMENT_DIFF > 90)` per applicant |
| **Implementation method** | Use the `PAYMENT_DIFF` computed in E3. Count exceedances per applicant. Also create binary flags: `EVER_DPD30`, `EVER_DPD60`, `EVER_DPD90`. |

---
## Group F — POS & Cash Balance Profile

The `POS_CASH_balance` table contains monthly snapshots of all active point-of-sale and cash loans. This captures how the borrower is currently managing their existing Horizon debt.

---

### Feature F1: Number of Active POS/Cash Contracts

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower juggling multiple simultaneous POS/cash loans may be over-leveraged. The count of active contracts is a direct measure of concurrent debt. |
| **Mathematical definition** | `POS_CONTRACT_COUNT = nunique(SK_ID_PREV)` per applicant (from the most recent month's snapshot) |
| **Implementation method** | Group by `SK_ID_CURR`, count unique contract IDs. Only include contracts that are still active (not fully paid). |

---

### Feature F2: Average Remaining Debt on Active Contracts

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower with high remaining debt on existing POS/cash loans may have less capacity to take on new debt. This is a real-time leverage measure. |
| **Mathematical definition** | `POS_AVG_REMAINING = mean(CNT_INSTALMENT_FUTURE)` — remaining installments. Or `POS_TOTAL_REMAINING_AMT = sum(AMT_INSTALMENT × CNT_INSTALMENT_FUTURE)` |
| **Implementation method** | Use the most recent month's snapshot per contract. `CNT_INSTALMENT_FUTURE` = remaining installments. Multiply by installment amount for total remaining balance. |

---

### Feature F3: POS Contract Delinquency Rate

| Aspect | Detail |
|---|---|
| **Business intuition** | The proportion of months where the borrower was delinquent (late on payment) across all POS/cash contracts. Consistent delinquency is a strong risk signal. |
| **Mathematical definition** | `POS_DELINQUENT_MONTHS = count(SKP_DPD > 0)` grouped by `SK_ID_CURR`. `POS_DELINQUENT_RATE = POS_DELINQUENT_MONTHS / total months observed` |
| **Implementation method** | `SKP_DPD` is days past due in the current month. Count months where `SKP_DPD > 0`. Normalize by total months observed (rows per applicant). |

---

### Feature F4: Maximum DPD Across POS Contracts

| Aspect | Detail |
|---|---|
| **Business intuition** | The single worst delinquency episode on any POS/cash contract captures the extreme tail of payment behavior. A borrower who fell 60 days behind at any point is qualitatively riskier. |
| **Mathematical definition** | `POS_MAX_DPD = max(SKP_DPB)` per applicant (DPD = days past due; DPB = days past due + days beyond? Use `SKP_DPD` and/or `SKP_DPB`). |
| **Implementation method** | Take max of both `SKP_DPD` and `SKP_DPB` per applicant across all months and contracts. |

---
## Group G — Credit Card Balance Profile

The `credit_card_balance` table contains monthly snapshots of all active credit card accounts at Home Credit. Credit cards are revolving products — utilization and payment behavior are key signals.

---

### Feature G1: Number of Credit Cards

| Aspect | Detail |
|---|---|
| **Business intuition** | Multiple credit cards indicate access to revolving credit. More cards = more opportunities to overspend. |
| **Mathematical definition** | `CC_COUNT = nunique(SK_ID_PREV)` per applicant |
| **Implementation method** | Group by `SK_ID_CURR`, count unique credit card contract IDs. |

---

### Feature G2: Total Credit Limit Across All Cards

| Aspect | Detail |
|---|---|
| **Business intuition** | High total credit limit suggests the borrower has been deemed creditworthy by other lenders (or by Horizon previously). Very high limits could also mean the borrower is a heavy credit user. |
| **Mathematical definition** | `CC_TOTAL_LIMIT = sum(AMT_CREDIT_LIMIT_ACTUAL)` per applicant (most recent month per contract) |
| **Implementation method** | For each contract, take the latest `MONTHS_BALANCE` snapshot. Sum `AMT_CREDIT_LIMIT_ACTUAL` across contracts. |

---

### Feature G3: Total Balance Carried on Cards

| Aspect | Detail |
|---|---|
| **Business intuition** | The balance is the amount the borrower owes on each card. High balances relative to limits indicate heavy usage and potential stress. |
| **Mathematical definition** | `CC_TOTAL_BALANCE = sum(AMT_BALANCE)` per applicant (most recent month per contract) |
| **Implementation method** | Same as G2: latest snapshot, sum `AMT_BALANCE`. |

---

### Feature G4: Credit Card Utilization Rate

| Aspect | Detail |
|---|---|
| **Business intuition** | Utilization = balance / limit. This is one of the most powerful credit risk signals in consumer finance (second only to payment history). Utilization > 80% is a strong distress signal. > 100% means over-limit (very high risk). |
| **Mathematical definition** | `CC_UTILIZATION = CC_TOTAL_BALANCE / CC_TOTAL_LIMIT`. If limit is 0, utilization is undefined (set to NaN). |
| **Implementation method** | Cap at [0, 2]. Create separate flags: `CC_OVER_LIMIT = 1 if utilization > 1 else 0`. |

---

### Feature G5: Minimum Payment Ratio

| Aspect | Detail |
|---|---|
| **Business intuition** | Borrowers who pay only the minimum on their credit cards are stretching their finances. Those who pay the full statement balance (or more) are financially healthier. The ratio of actual payment to minimum payment captures this. |
| **Mathematical definition** | `CC_MIN_PAY_RATIO = mean(AMT_PAYMENT_CURRENT / AMT_INST_MIN_REGULARITY)` per applicant |
| **Implementation method** | For each month (most recent snapshots), divide actual payment by minimum required. Take average across months and cards. Values > 1 = paying more than minimum (good). Values ≈ 1 = paying only minimum (okay). Values < 1 = paying less than minimum (bad — falling behind). |

---

### Feature G6: Credit Card DPD

| Aspect | Detail |
|---|---|
| **Business intuition** | The same DPD logic from POS applies to credit cards. Amex/Discover/Visa cards are the most watched credit products — delinquency here is a very strong signal. |
| **Mathematical definition** | `CC_MAX_DPD = max(SKP_DPD)` and `CC_AVG_DPD = mean(SKP_DPD)` per applicant |
| **Implementation method** | Take max and mean across all months and all card contracts. Binary flag: `CC_EVER_LATE = 1 if CC_MAX_DPD > 0 else 0`. |

---
## Group H — Categorical Encoding

All categorical variables must be converted to numeric form. Different encoding strategies are appropriate for different cardinalities and ordinality.

---

### Feature H1: Ordinal Encoding

| Aspect | Detail |
|---|---|
| **Business intuition** | Some categories have a natural ordering. Preserving that order lets the model learn monotonic risk trends. |
| **Mathematical definition** | Map each level to an integer preserving order. |
| **Implementation method** | 
- `NAME_EDUCATION_TYPE`: Lower secondary → 1, Secondary / secondary special → 2, Incomplete higher → 3, Higher education → 4, Academic degree → 5
- `NAME_FAMILY_STATUS`: Single / not married → 1, Separated → 2, Divorced → 3, Widow → 4, Married → 5
- `NAME_HOUSING_TYPE`: With parents → 1, Rented → 2, Office apartment → 3, Municipal apartment → 4, Co-op apartment → 5, House / apartment → 6
- `WEEKDAY_APPR_PROCESS_START`: ordered by day of week (Monday=1 ... Sunday=7)
|

### Feature H2: One-Hot Encoding (Low-Cardinality Nominal)

| Aspect | Detail |
|---|---|
| **Business intuition** | For categories with no ordering (gender, contract type), one-hot encoding allows the model to learn unique risk contributions per category. |
| **Mathematical definition** | Create k binary columns for k categories. Drop the first to avoid multicollinearity (dummy trap). |
| **Implementation method** | Apply `pd.get_dummies(drop_first=True)` or `sklearn.preprocessing.OneHotEncoder(drop='first')`. Applicable to: `CODE_GENDER`, `NAME_CONTRACT_TYPE`, `NAME_INCOME_TYPE`, `OCCUPATION_TYPE` (if cardinality is manageable), `ORGANIZATION_TYPE` (high cardinality — see H3). |

### Feature H3: Frequency Encoding (High-Cardinality Nominal)

| Aspect | Detail |
|---|---|
| **Business intuition** | Columns like `ORGANIZATION_TYPE` have 50+ categories. One-hot would create too many sparse columns. Frequency encoding replaces each category with its prevalence, capturing the idea that rare employer types may be riskier (less stable / niche industries). |
| **Mathematical definition** | `FREQ_ENCODED = count(category) / total_rows` for each category |
| **Implementation method** | Compute category frequencies from training data only. Map to each row in train and test. Apply to: `ORGANIZATION_TYPE`, `OCCUPATION_TYPE`, `NAME_TYPE_SUITE`, `NAME_PRODUCT_TYPE`. |

### Feature H4: Target Encoding (with Cross-Validation Safeguard)

| Aspect | Detail |
|---|---|
| **Business intuition** | Target encoding replaces each category with the mean default rate for that category. This captures the pure risk association. However, it risks target leakage if not done carefully. |
| **Mathematical definition** | `TARGET_ENCODED = mean(TARGET | category)` using k-fold cross-validation: for each fold, compute the mean on the training folds only, apply to the held-out fold. Add global prior for regularization. |
| **Implementation method** | Use `category_encoders.TargetEncoder` or implement manually with `StratifiedKFold`. Apply a smoothing factor: `encoded = (n_cat × mean_cat + m × global_mean) / (n_cat + m)` where `m` is a smoothing parameter. Only apply to categories with known risk association (e.g., `OCCUPATION_TYPE` has clear risk tiers). |

---
## Group I — Missing-Value Indicators

### Purpose
Missingness in credit data is rarely random. A missing `EXT_SOURCE_2` means the bureau had insufficient data on this borrower — which itself signals thin credit history (higher risk). We preserve this signal via binary indicator features.

---

### Feature I1: Missing-Indicator Flags

| Aspect | Detail |
|---|---|
| **Business intuition** | For every feature with non-trivial missingness (>2%), create a binary flag: `COLNAME_NA = 1 if COLNAME is NaN else 0`. This lets the model learn the risk premium associated with missingness. |
| **Mathematical definition** | `isnan(feature).astype(int)` |
| **Implementation method** | Iterate over features, check `missing_ratio > 0.02`, create flag. Candidate features: `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`, `OWN_CAR_AGE`, `AMT_REQ_CREDIT_BUREAU_*`, `AMT_ANNUITY` (some missing), `CNT_FAM_MEMBERS`. |

### Feature I2: Imputation Strategy

| Aspect | Detail |
|---|---|
| **Feature** | **Imputation Value** | **Rationale** |
|---|---|---|
| `EXT_SOURCE_1/2/3` | Median (after flagging missing) | Median preserves distribution center. Missing flag captures the thin-file effect. |
| `AMT_ANNUITY` | Median (by `AMT_CREDIT` decile) | Annuity depends on loan amount; median within amount decile is more accurate than global median. |
| `AMT_INCOME_TOTAL` | Median (by `NAME_EDUCATION_TYPE`) | Income correlates with education; group median preserves the relationship. |
| `DAYS_EMPLOYED` (after artifact removal) | Median (by `NAME_EDUCATION_TYPE`) | Employment length varies by education level. |
| `OWN_CAR_AGE` | 0 (assume no car) | Most missing values likely mean no car. The `HAS_CAR` flag captures the missingness signal. |
| `CNT_FAM_MEMBERS` | 2 (median) | Default to a 2-person household (most common in the data). |
| Categorical variables | 'Unknown' category | Preserves interpretability; the model learns the risk of 'Unknown' separately. |

---
## Group J — Interaction Features

### Purpose
Credit risk is often non-linear and interactive. A young borrower with a high DTI is far riskier than either signal alone. Interactions let the model capture these compounding effects without the model having to discover them from scratch.

---

### Feature J1: EXT_SOURCE_2 × EXT_SOURCE_3 (Credit Score Interaction)

| Aspect | Detail |
|---|---|
| **Business intuition** | These two external scores measure similar but distinct concepts (different credit bureaus). A borrower who scores low on *both* is qualitatively worse than the additive sum of their scores would suggest — they are being flagged as high-risk by two independent assessors. |
| **Mathematical definition** | `EXT_2x3 = EXT_SOURCE_2 × EXT_SOURCE_3` |
| **Implementation method** | Multiply the two scores (after imputation). Standardize the result since the product will have a different scale than either input. Handle missing: if either is missing, the interaction is missing (flag in Group I). |

### Feature J2: DTI × EXT_SOURCE_2 (Burden × Creditworthiness)

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower with both high DTI (overextended) and low external score (poor credit history) is in a dangerous zone. The interaction amplifies the risk beyond what either feature captures independently. |
| **Mathematical definition** | `DTI_x_EXT2 = DTI × EXT_SOURCE_2` |
| **Implementation method** | Standardize DTI and EXT_SOURCE_2 before multiplying (to avoid one term dominating). |

### Feature J3: Age × Income (Life-Stage Financial Capacity)

| Aspect | Detail |
|---|---|
| **Business intuition** | A 25-year-old earning $80K is in a very different financial position from a 55-year-old earning $80K. The older borrower likely has more savings and assets, hence lower risk. The interaction captures life-stage-adjusted earning power. |
| **Mathematical definition** | `AGE_INCOME = AGE_YEARS × AMT_INCOME_TOTAL` |
| **Implementation method** | Standardize both features first, then multiply. |

### Feature J4: LTI × External Score (Exposure × Credit Quality)

| Aspect | Detail |
|---|---|
| **Business intuition** | A borrower taking a loan that is 10× their annual income is risky — unless their external score is very high, indicating strong credit history that justifies the higher exposure. This interaction separates risky high-LTI borrowers from creditworthy ones. |
| **Mathematical definition** | `LTI_x_EXT2 = LTI × EXT_SOURCE_2` |
| **Implementation method** | Standardize both, multiply. |

### Feature J5: Employment Length × Age (Career Stability Index)

| Aspect | Detail |
|---|---|
| **Business intuition** | The ratio A6 captures employment length fraction. The interaction `AGE × EMP_LENGTH` gives absolute years of experience, which the model can use differently from the fraction. A 50-year-old with 25 years of employment is very stable. |
| **Mathematical definition** | `CAREER_INDEX = AGE_YEARS × EMP_LENGTH_YEARS` |
| **Implementation method** | Multiply cleaned years (after artifact removal). Standardize. |

---
## Group K — Feature Selection

### Purpose
After aggregation, we expect 200–300 features. Not all will be useful. Feature selection reduces dimensionality, improves generalization, speeds up training, and simplifies deployment.

---

### Step K1: Zero-Variance & Near-Zero-Variance Removal

| Aspect | Detail |
|---|---|
| **Business intuition** | A feature that is constant or nearly constant (e.g., 99.9% of values are 0) cannot discriminate between defaulters and repayers. |
| **Mathematical definition** | Drop features where `variance < threshold` (e.g., 0.01) or where a single value accounts for >99% of observations. |
| **Implementation method** | Use `sklearn.feature_selection.VarianceThreshold`. Fit on training data only. |

### Step K2: High-Correlation Pair Removal

| Aspect | Detail |
|---|---|
| **Business intuition** | Two features that are perfectly correlated (e.g., total debt and average debt × count) carry redundant information. Keeping both adds noise without predictive benefit. |
| **Mathematical definition** | For any pair with |Pearson r| > 0.90, drop the feature with lower absolute correlation to the target. |
| **Implementation method** | Compute correlation matrix. Flag pairs above threshold. For each pair, keep the one with higher `abs(corr_with_target)`. |

### Step K3: Mutual Information Filtering

| Aspect | Detail |
|---|---|
| **Business intuition** | Mutual information captures any relationship (linear or non-linear) between a feature and the target. It is a more powerful filter than Pearson correlation. |
| **Mathematical definition** | `MI(X, Y) = sum[ p(x,y) × log(p(x,y) / (p(x) × p(y))) ]` |
| **Implementation method** | Use `sklearn.feature_selection.mutual_info_classif`. Sort features by MI score. Keep top K features (e.g., top 150) as a pre-filter before model-based selection. Use 5-fold MI estimates to reduce variance. |

### Step K4: L1-Regularization (LASSO) Feature Selection

| Aspect | Detail |
|---|---|
| **Business intuition** | L1-regularized logistic regression drives coefficients of irrelevant features to exactly zero. The features with non-zero coefficients are the ones the model found genuinely predictive. |
| **Mathematical definition** | Minimize `LogLoss + lambda × sum(|beta_i|)`. Features with `beta_i = 0` are eliminated. |
| **Implementation method** | Train `LogisticRegression(penalty='l1', solver='saga', C=0.1)` on the training set. Extract non-zero coefficient features. Tune `C` (inverse of lambda) via cross-validation. |

### Step K5: Final Feature Set Assembly

| Aspect | Detail |
|---|---|
| **Business intuition** | We want a lean, robust feature set that captures all major risk dimensions without redundancy. |
| **Mathematical definition** | Intersection or union of selected feature sets from K3 and K4. |
| **Implementation method** | Take the union of: (a) top-150 by mutual information, (b) non-zero LASSO features. Ensure at least 2 features from each feature group (A–J) are retained for diversity. Document the final feature count and which groups contributed most. |

---
## Summary: Feature Engineering Pipeline

### Execution Order

```
Start: Raw tables (8 files)
   │
   ├── Step 1: Load & validate primary keys (SK_ID_CURR consistency)
   │
   ├── Step 2: Clean application_train
   │   ├── B1: DAYS_EMPLOYED anomaly flag
   │   ├── B2: Clean DAYS_EMPLOYED
   │   ├── B3: Cap income (P99.9)
   │   ├── B4: Flag extreme loans
   │   └── I2: Impute basic columns
   │
   ├── Step 3: Create domain ratio features (Group A)
   │   ├── A1: DTI
   │   ├── A2: LTI
   │   ├── A3: Income per household
   │   ├── A4: Annuity rate
   │   ├── A5: EXT composites
   │   ├── A6: Employment ratio
   │   ├── A7: Age bins
   │   └── A8: Car age
   │
   ├── Step 4: Aggregate bureau table → 1 row per applicant (Group C)
   ├── Step 5: Aggregate bureau_balance → join to bureau agg (C8)
   ├── Step 6: Aggregate previous_application (Group D)
   ├── Step 7: Aggregate installments_payments (Group E)
   ├── Step 8: Aggregate POS_CASH_balance (Group F)
   ├── Step 9: Aggregate credit_card_balance (Group G)
   │
   ├── Step 10: Merge all aggregations into master DataFrame
   ├── Step 11: Categorical encoding (Group H)
   ├── Step 12: Create interaction features (Group J)
   ├── Step 13: Create missing indicators + impute (Group I)
   │
   ├── Step 14: Feature selection (Group K)
   │
   └── Output: Master DataFrame (~150–250 features) ready for modeling
```

### Anti-Patterns to Avoid

| Anti-Pattern | Why It Is Dangerous | Mitigation |
|---|---|---|
| Target leakage from future data | Some bureau/previous features may contain information from AFTER the application date | Filter all temporal features: only include bureau data where `DAYS_CREDIT >= DAYS_APPLICATION`? (Check Home Credit data dictionary carefully) |
| Fit imputation scalers on full data before train/test split | Mean/median from test set leaks into training, inflating performance | Fit all scalers, imputers, and encoders on training data only; transform test data |
| One-hot encoding rare categories | Creates columns with near-zero variance that add noise | Group rare categories (<0.1% of rows) into 'Other' before one-hot |
| Target encoding without cross-validation | Direct target mean per category creates extreme overfitting | Always use k-fold CV target encoding with smoothing |
| Feature selection using target correlation on the full dataset | Peeking at test set to select features overestimates model performance | Apply feature selection inside cross-validation folds |

---
## Next Steps

1. **Implement `src/preprocess.py`** — translate this design document into a reusable preprocessing pipeline that covers all 14 steps above.
2. **Apply the pipeline** to produce the master training DataFrame.
3. **Proceed to Model Building** (`notebooks/04_model_building.ipynb`) using the engineered features.

**Total expected feature count:** 150–250 (after selection from ~300 raw engineered features).

**Key verification step before modeling:** Confirm that the test set pipeline produces the same columns as the training pipeline. Use `assert train.columns.equals(test.columns)`.